# 01 Attention 机制概述

前面我们已经完成了 PyTorch、MLP、CNN 和 CNN-MNIST 实战。

现在开始进入 Attention，也就是 **注意力机制**。

这一节先不写代码，也不直接进入 Transformer。我们先像前面学 CNN 一样，把问题讲清楚：

```text
Attention 为什么会出现？
它到底在解决什么问题？
它的计算过程为什么是这样一步步推出来的？
它和前面学过的 MLP、CNN、Softmax 有什么关系？
```

这一节的目标不是背公式，而是先建立一条清楚的主线：

```text
信息很多 -> 重要程度不同 -> 先算相关性分数 -> 用 Softmax 变成权重 -> 按权重汇总信息
```

## 1. 为什么学完 CNN 后要学 Attention

前面学 CNN 时，我们一直在解决图像分类问题。

以 MNIST 为例，任务很明确：

```text
输入：一张 1 x 28 x 28 的手写数字图片
输出：0 到 9 中的一个类别
```

CNN 的核心思想是：

```text
先看局部区域，提取边缘、笔画、局部形状，再逐层组合成整体特征。
```

这套思路非常适合图像，因为图像里的相邻像素通常关系很强。

但是深度学习不只处理图像分类。很多任务里，模型面对的是一串信息：

- 一句话里的多个词。
- 一段文本里的多个句子。
- 一张图片切成的多个区域。
- 一段时间序列里的多个时间点。

这时候模型经常要回答一个新问题：

```text
面对这么多位置的信息，当前这一步到底应该重点参考哪些位置？
```

Attention 就是围绕这个问题发展出来的。

## 2. 先看一个具体问题：信息不是同等重要的

假设有这样一句话：

```text
小明把复习资料交给小红，因为她明天要考试。
```

如果要判断“她”指的是谁，我们不会把每个词都平均看待。

我们会更关注：

```text
小红
明天要考试
复习资料
```

而不是平均地把“把”“给”“因为”这些词也看得同样重要。

再换一个问题，如果现在问“交出去的东西是什么”，重点又会变成：

```text
复习资料
交给
```

注意这里有一个很关键的点：

```text
同一句话，在不同问题下，重要信息会变化。
```

这就是 Attention 的背景：模型不能只会“整体处理”，还要能根据当前需要，动态决定重点看哪里。

## 3. 传统做法为什么会遇到困难

先不要急着说 Attention，我们先看没有 Attention 时会有什么麻烦。

### 3.1 MLP 的问题

MLP 的 Linear 层会学习一组固定参数：

$$
\mathbf{y}=\mathbf{W}\mathbf{x}+\mathbf{b}
$$

训练完成后，$\mathbf{W}$ 和 $\mathbf{b}$ 会保存下来。不同样本进来时，模型使用的是同一组参数。

这并不是说 MLP 没有表达能力，而是说它的权重本身不是针对“这一次输入里谁和谁更相关”临时算出来的。

### 3.2 CNN 的问题

CNN 擅长从局部窗口开始提取特征：

```text
小窗口 -> 局部特征 -> 更深层组合成更大范围的特征
```

随着网络加深，CNN 的感受野会变大，所以 CNN 不是只能看局部。

但它建立远距离关系通常需要经过多层传播。对于文本或长序列来说，如果两个位置相距很远，模型想让它们直接建立联系就不那么自然。

### 3.3 序列任务里的压缩问题

在早期机器翻译等序列任务中，常见做法是先把整句话编码成一个向量，再用这个向量生成输出。

这就像把一本书压缩成一句摘要，然后后面所有问题都只能靠这句摘要回答。

句子很短时还能勉强工作，句子一长，很多细节就容易丢失。

Attention 的想法就是：

```text
不要只依赖一个压缩后的整体向量。
每一步需要信息时，都可以回头看看输入里的各个位置，并决定重点参考谁。
```

## 4. Attention 的全称和本质

Attention 的常见完整说法是 **Attention Mechanism**，中文叫 **注意力机制**。

这里的“机制”两个字很重要。

Attention 通常不是单独拿来当一个完整模型，而是神经网络里面的一种信息汇总方法。

它做的事情可以先用一句话概括：

```text
根据当前需要，为不同信息分配不同权重，再按权重把信息汇总起来。
```

所以 Attention 不是让模型像人一样真的产生意识，也不是玄乎地“看一眼哪里重要”。

它本质上是一套可计算、可训练、可反向传播的加权汇总过程。

## 5. 从最简单的“平均汇总”开始推导

假设现在有三份信息，用数字先简单表示：

```text
信息 A = 10
信息 B = 20
信息 C = 30
```

如果不知道谁更重要，最简单的做法是求平均：

$$
\frac{10+20+30}{3}=20
$$

平均的意思是：每一份信息的重要程度一样。

也可以写成加权求和：

$$
\frac{1}{3}\times10+\frac{1}{3}\times20+\frac{1}{3}\times30=20
$$

这里每个权重都是 $\frac{1}{3}$。

但是实际任务里，经常不是每份信息都一样重要。

如果当前更需要参考 B，那么权重可以变成：

```text
A 的权重：0.1
B 的权重：0.7
C 的权重：0.2
```

这时汇总结果变成：

$$
0.1\times10+0.7\times20+0.2\times30=21
$$

结果更靠近 B，因为 B 的权重最大。

这一步已经接近 Attention 的核心了：

```text
不是把所有信息平均混在一起，而是让更相关的信息占更大比例。
```

## 6. 权重需要满足什么条件

如果我们要把权重理解成“每份信息参与汇总的比例”，通常希望它满足两个条件。

第一个条件：权重不能是负数。

$$
\alpha_i\geq0
$$

第二个条件：所有权重加起来等于 1。

$$
\sum_i\alpha_i=1
$$

这样一来，最终结果就可以写成：

$$
\mathbf{c}=\sum_i\alpha_i\mathbf{v}_i
$$

这里先解释符号：

- $\mathbf{v}_i$ 表示第 $i$ 份真正被汇总的信息。
- $\alpha_i$ 表示第 $i$ 份信息的注意力权重。
- $\mathbf{c}$ 表示最后汇总出来的新信息。

这条公式非常重要。

它说明 Attention 最后的输出不是凭空产生的，而是由原来的多份信息按不同权重组合出来的。

## 7. 问题来了：权重从哪里来

刚才我们直接写出了权重：

```text
[0.1, 0.7, 0.2]
```

但模型不能靠我们手工告诉它谁重要。

所以真正的问题是：

```text
模型如何自己算出这些权重？
```

思路分两步。

第一步，先算每份信息和当前需求的相关程度，得到一组原始分数。

```text
信息 A 的相关性分数：1.0
信息 B 的相关性分数：3.0
信息 C 的相关性分数：2.0
```

第二步，把这些原始分数转换成可以用于加权求和的比例。

这就需要用到前面学过的 Softmax。

## 8. Softmax 为什么刚好适合做这件事

前面学习多分类时，Softmax 的作用是：

```text
把一组原始类别分数变成一组概率。
```

在 Attention 里，Softmax 的作用非常类似：

```text
把一组相关性分数变成一组注意力权重。
```

假设相关性分数是：

$$
\mathbf{s}=[1.0,3.0,2.0]
$$

Softmax 的公式是：

$$
\alpha_i=\frac{\exp(s_i)}{\sum_j \exp(s_j)}
$$

这里可以这样读：

$$
s_i：第 i 个原始相关性分数
$$

$$
exp(s_i)：对这个分数取指数
$$

$$
\alpha_i：Softmax 后得到的注意力权重
$$

一步步算：

$$
\exp(1.0)\approx2.718
$$

$$
\exp(3.0)\approx20.086
$$

$$
\exp(2.0)\approx7.389
$$

总和是：

$$
2.718+20.086+7.389=30.193
$$

所以权重大约是：

$$
\alpha=[0.090,0.665,0.245]
$$

这组权重满足两个条件：

```text
每个权重都大于 0
所有权重加起来等于 1
```

所以它们可以拿来做加权求和。

## 9. 把分数、权重、汇总结果串起来

现在完整走一遍。

原始信息是：

```text
A = 10
B = 20
C = 30
```

模型先算相关性分数：

```text
score = [1.0, 3.0, 2.0]
```

经过 Softmax 得到注意力权重：

```text
alpha = [0.090, 0.665, 0.245]
```

最后加权求和：

$$
0.090\times10+0.665\times20+0.245\times30\approx21.55
$$

现在就可以把 Attention 的最小流程写出来：

```text
第 1 步：算相关性分数
第 2 步：用 Softmax 把分数变成权重
第 3 步：按照权重对信息做加权求和
```

更短地写就是：

```text
相关性分数 -> Softmax -> 注意力权重 -> 加权汇总
```

## 10. 为什么说 Attention 是“动态”的

Attention 里的权重不是一组固定写死的数字。

同一句话，在不同问题下会得到不同权重。

还是刚才那句话：

```text
小明把复习资料交给小红，因为她明天要考试。
```

如果当前问题是“她指的是谁”，模型应该更关注人物关系。

如果当前问题是“交出去的东西是什么”，模型应该更关注“复习资料”。

同一份输入，因为当前需求不同，相关性分数就可能不同，Softmax 后的权重也就不同。

这就是动态的意思：

```text
注意力权重会根据当前输入和当前需求临时计算出来。
```

## 11. Attention 和 Linear 层的权重有什么区别

这里很容易混淆。

前面 MLP 里的 Linear 层也有权重，比如：

$$
\mathbf{y}=\mathbf{W}\mathbf{x}+\mathbf{b}
$$

这里的 $\mathbf{W}$ 是模型参数。训练时它会更新，训练后它会保存下来。

Attention 里的注意力权重 $\alpha$ 不一样。

$\alpha$ 通常不是直接保存下来的模型参数，而是根据当前输入临时算出来的结果。

可以这样对比：

```text
Linear 的 W：训练学到的固定参数。
Attention 的 alpha：本次输入计算出来的动态权重。
```

那 Attention 里面有没有可学习参数呢？

有。

后面学习 Query、Key、Value 时，会看到模型会学习一些线性变换参数，用它们生成用于匹配和汇总的表示。

但这一节先记住核心区别：

```text
注意力权重本身是动态计算结果，不是固定保存的一张参数表。
```

## 12. Attention 和 CNN 的关注方式有什么不同

CNN 和 Attention 都在处理“哪些信息重要”的问题，但方式不同。

CNN 的思路是从局部开始：

```text
卷积核在局部窗口里滑动，先提取附近像素组成的局部特征。
```

比如识别手写数字时，CNN 很适合先找边缘、横线、竖线、弯曲笔画。

Attention 的思路是让位置之间直接计算关系：

```text
当前位置可以直接判断输入中其他位置和自己有多相关。
```

所以可以先这样理解：

```text
CNN：更强调从局部区域提取特征，再逐层组合。
Attention：更强调根据当前内容，直接给不同位置分配权重。
```

这不是说 CNN 不好，也不是说 Attention 永远更强。

它们只是组织信息的方式不同。CNN 的局部归纳偏置很适合图像；Attention 的动态关系建模很适合需要长距离依赖和全局交互的场景。

## 13. 用形状理解 Attention 在处理什么

后面学习 Attention 时，经常会看到输入形状：

$$
B\times N\times D
$$

其中：

- $B$ 表示 batch size，一批里有多少个样本。
- $N$ 表示一个样本里有多少个位置，比如一句话有多少个词，或者一张图片被切成多少个区域。
- $D$ 表示每个位置用多少维特征来表示。

Attention 重点处理的是 $N$ 这个维度上的关系。

也就是说，它会思考：

```text
第 1 个位置应该参考哪些位置？
第 2 个位置应该参考哪些位置？
第 3 个位置应该参考哪些位置？
...
```

如果一个样本有 $N$ 个位置，那么每个位置都可能去看其他 $N$ 个位置。

所以 Attention 中常见的注意力权重可以理解成一个 $N\times N$ 的关系表。

```text
每一行：当前这个位置看其他位置的权重分布。
每一列：某个位置被其他位置参考的程度。
```

## 14. 先提前认识一下 Q、K、V，但不展开

后面学习标准 Attention 时，会经常看到三个字母：

```text
Q：Query
K：Key
V：Value
```

这一节先不深入，只先给一个直觉。

可以暂时这样理解：

```text
Query：当前我想找什么。
Key：每份信息拿什么来和 Query 匹配。
Value：真正被加权汇总的内容。
```

于是 Attention 的完整味道就出来了：

```text
用 Query 和 Key 算相关性分数。
用 Softmax 把分数变成权重。
用权重对 Value 加权求和。
```

对应成公式就是：

$$
\text{Attention}=\sum_i \alpha_i\mathbf{v}_i
$$

其中 $\alpha_i$ 来自 Query 和 Key 的相关性分数。

下一节我们再专门拆 Q、K、V。现在先不要急着把它们背成三个孤立名词。

## 15. Attention 也有代价

Attention 很强，但不是没有成本。

如果一个输入有 $N$ 个位置，每个位置都去和其他位置计算关系，那么关系数量大约是：

$$
N\times N
$$

这说明序列越长，计算量和显存压力会增长得很快。

另外，单独的 Attention 只是根据内容计算关系，它并不天然知道“第一个词在前、第二个词在后”。

所以后面学习 Transformer 时，还会遇到两个重要内容：

```text
为什么 Attention 计算量比较大
为什么需要位置编码
```

这一节先记住：Attention 是一种强大的动态信息汇总方法，但它也有自己的计算代价和结构限制。

## 16. 本节小结

这一节先记住下面这条主线：

```text
输入里有很多信息。
不同信息对当前任务的重要程度不同。
模型先计算当前需求和每份信息的相关性分数。
Softmax 把相关性分数变成非负、总和为 1 的权重。
模型再按照权重把信息加权汇总起来。
```

核心公式可以先记成两步：

$$
\alpha_i=\operatorname{softmax}(s_i)
$$

$$
\mathbf{c}=\sum_i\alpha_i\mathbf{v}_i
$$

更口语一点说：

```text
先判断谁更相关，再按相关程度汇总信息。
```

这就是 Attention 的第一层理解。

## 17. 自测问题

1. 为什么学完 CNN 后，处理序列或多位置输入时还需要学习 Attention？
2. 为什么不能总是把所有信息平均汇总？
3. Attention Mechanism 的中文含义是什么？为什么说它是一种机制，而不是一个完整模型？
4. 注意力权重通常需要满足哪两个条件？
5. 相关性分数和注意力权重有什么区别？
6. Softmax 在 Attention 中的作用是什么？
7. 为什么说 Attention 的权重是动态计算出来的？
8. Linear 层里的权重和 Attention 里的注意力权重有什么不同？
9. CNN 和 Attention 在组织信息时分别更强调什么？
10. `B x N x D` 中的 `N` 为什么对 Attention 特别重要？
11. Q、K、V 可以先分别理解成什么？
12. Attention 为什么会有 `N x N` 级别的计算代价？